# B1 Method — Convergent Basis Vector Derivation

**Domain-independent inference of canonical basis vectors from K independent sources.**

Upload your alignment CSV (candidates × sources) and optional sources metadata CSV, then run the analysis.

---

**Also available as:**
- `pip install b1-method` — Python package with CLI ([PyPI](https://pypi.org/project/b1-method/))
- [b1-method.streamlit.app](https://b1-method.streamlit.app) — browser UI (drag & drop)
- [GitHub](https://github.com/capraCoder/AIBOS_B1) — source code

**Reference:** Caprazli, K. M. (2026). *Specifying Order from Geometry: A Domain-Independent Method for Inferring Canonical Basis Vectors from Heterogeneous Observations.*

In [ ]:
# Install the b1-method package (runs once)
!pip install -q b1-method

## Option A: Upload your own CSVs

**Alignment CSV format:**
```
candidate,Source1,Source2,Source3
Extraversion,Y,Y,Y*
Agreeableness,Y,N,Y
```
Values: `Y` (present), `Y*` (present with caveats), `N` (absent), `N*` (absent with caveats)

**Sources CSV format (optional):**
```
name,method_type,languages,tradition,year
Source1,lexical FA,English,lexical,1990
Source2,questionnaire,English;German,questionnaire,1992
```

In [ ]:
from google.colab import files
import os

print("Upload your alignment CSV (required):")
uploaded = files.upload()
alignment_file = list(uploaded.keys())[0]
print(f"  → {alignment_file}")

print("\nUpload your sources CSV (optional — press Cancel to skip):")
try:
    uploaded2 = files.upload()
    sources_file = list(uploaded2.keys())[0] if uploaded2 else None
    print(f"  → {sources_file}")
except Exception:
    sources_file = None
    print("  → Skipped (no source metadata)")

In [ ]:
from b1_method.core import B1Analysis
from b1_method.io import load_alignment_csv, load_sources_csv

# Load data
alignment, source_names = load_alignment_csv(alignment_file)
sources = load_sources_csv(sources_file) if sources_file else None

# Run analysis
domain = input("Domain name (e.g. Personality, Emotion): ") or "Analysis"
result = B1Analysis(alignment, sources=sources, domain=domain).run()

# Print report
B1Analysis.print_report(result)

In [ ]:
# Download JSON report
from b1_method.io import save_report_json
from google.colab import files as colab_files

out_path = f"b1_report_{domain.lower().replace(' ', '_')}.json"
save_report_json(result, out_path)
print(f"Report saved: {out_path}")
colab_files.download(out_path)

---
## Option B: Try with bundled examples

Run one of the four domains from the paper (no upload needed).

In [ ]:
import importlib.resources
from b1_method.core import B1Analysis
from b1_method.io import load_alignment_csv, load_sources_csv

# Pick a domain: personality, emotion, brand, intelligence
DOMAIN = "personality"  # ← change this

examples_dir = importlib.resources.files("b1_method") / "examples"
align_path = str(examples_dir / f"{DOMAIN}_alignment.csv")
src_path = str(examples_dir / f"{DOMAIN}_sources.csv")

alignment, source_names = load_alignment_csv(align_path)
sources = load_sources_csv(src_path)

result = B1Analysis(alignment, sources=sources, domain=DOMAIN.title()).run()
B1Analysis.print_report(result)

---
## Temporal Holdout Simulation

Feed sources chronologically and watch B1's verdicts evolve. Requires sources CSV with `year` column.

In [ ]:
from b1_method.temporal import TemporalSimulation, print_temporal_report
from b1_method.io import load_alignment_csv, load_sources_csv

# Uses the example data loaded above (re-run Option B first if needed)
candidates = list(alignment.keys())
temporal_sources = []
for i, meta in enumerate(sources):
    factors = {c: alignment[c][i] if i < len(alignment[c]) else "N" for c in candidates}
    temporal_sources.append({**meta, "factors": factors})

sim = TemporalSimulation(
    domain_name=DOMAIN.title(),
    sources=temporal_sources,
    ground_truth={},
    candidates=candidates,
)
result_temporal = sim.run()
print_temporal_report(result_temporal)

---
*B1 Method v0.1.0 — Caprazli (2026) — [cosmologic.pro](https://cosmologic.pro)*